# omniASR on the 30 office-testing recordings

Transcribes the 30 Survey2026 office-testing recordings (Sep 17–21, 2026) with Meta's Omnilingual ASR, for comparison with the Whisper models in `CER_WER_calculation.ipynb`.

**Before running:** choose a GPU runtime (Runtime → Change runtime type). Pick **A100** (or L4); a **T4** also works but falls back to the 3B model.

Then use Runtime → Run all. At the end, `omniasr_transcripts.json` downloads. Put it in `ASR_Accuracy/office_testing/`.

In [ ]:
# 1. Check the GPU and pick the model
import torch
assert torch.cuda.is_available(), "No GPU - choose a GPU runtime first (Runtime > Change runtime type)"
gpu = torch.cuda.get_device_properties(0)
gpu_gb = gpu.total_memory / 1024**3
print(f"GPU: {gpu.name}, {gpu_gb:.1f} GB")

# omniASR-LLM-7B needs ~17 GiB in bf16; leave headroom for activations.
# Set MODEL_CARDS by hand to override, e.g. ["omniASR_LLM_7B", "omniASR_LLM_7B_v2"]
MODEL_CARDS = ["omniASR_LLM_7B"] if gpu_gb >= 22 else ["omniASR_LLM_3B"]
print("Will run:", MODEL_CARDS)

In [ ]:
# 2. Install Omnilingual ASR in its own Python 3.12 environment (a few minutes)
# omnilingual-asr needs fairseq2 0.5.2-0.6.0, which only has builds for PyTorch 2.7.1/2.8.0 on Python 3.10-3.12.
# Colab's own Python/PyTorch are newer, so this uses a separate environment and leaves Colab's untouched.
!apt-get -qq install -y libsndfile1 > /dev/null
!pip -q install uv
!uv venv -q --python 3.12 /content/omni
!uv pip install -q --python /content/omni/bin/python --index-strategy unsafe-best-match "torch==2.8.0" "torchaudio==2.8.0" "fairseq2[arrow]==0.6.0" omnilingual-asr --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.8.0/cu128
!/content/omni/bin/python -c "import torch, fairseq2, omnilingual_asr; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '| fairseq2', fairseq2.__version__)"

In [ ]:
# 3. Download the 30 recordings from GitHub (same entries as office_testing/expected_answers.json)
import json, os, urllib.request
RECORDINGS = [
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q8",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q8_20260917_124101.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q9",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q9_20260917_124248.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q10",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q10_20260917_124425.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q11",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q11_20260917_124543.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q12",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q12_20260917_125021.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q13",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q13_20260917_125119.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q14",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q14_20260917_125429.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q15",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q15_20260917_125546.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q16",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q16_20260917_125652.wav"
 },
 {
  "survey_id": "46ec4547-5bbe-4b02-9c2a-cbcec7223e2c",
  "question_id": "Q17",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-17/voice/voice_46ec4547-5bbe-4b02-9c2a-cbcec7223e2c_Q17_20260917_125950.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q8",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q8_20260921_113046.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q9",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q9_20260921_113118.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q10",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q10_20260921_113155.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q11",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q11_20260921_113235.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q12",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q12_20260921_113324.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q13",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q13_20260921_113403.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q14",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q14_20260921_113434.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q15",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q15_20260921_113516.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q16",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q16_20260921_113603.wav"
 },
 {
  "survey_id": "4f629181-d1dc-46cf-8e71-d819a4b33662",
  "question_id": "Q17",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_4f629181-d1dc-46cf-8e71-d819a4b33662_Q17_20260921_113640.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q8",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q8_20260921_053701.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q9",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q9_20260921_053808.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q10",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q10_20260921_054007.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q11",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q11_20260921_054045.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q12",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q12_20260921_054121.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q13",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q13_20260921_054145.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q14",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q14_20260921_054214.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q15",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q15_20260921_054249.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q16",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q16_20260921_054346.wav"
 },
 {
  "survey_id": "7db59ab6-e13d-4080-bb07-3842d8b51fe4",
  "question_id": "Q17",
  "url": "https://raw.githubusercontent.com/ishizuki-tech/SurveyExports/main/2026-09-21/voice/voice_7db59ab6-e13d-4080-bb07-3842d8b51fe4_Q17_20260921_054455.wav"
 }
]
os.makedirs("wavs", exist_ok=True)
for r in RECORDINGS:
    r["path"] = os.path.join("wavs", r["url"].rsplit("/", 1)[-1])
    if not os.path.exists(r["path"]):
        urllib.request.urlretrieve(r["url"], r["path"])
print(f"{len(RECORDINGS)} recordings ready")

In [ ]:
# 4. Transcribe in Swahili (swh_Latn) using the omniASR environment, and save the results
SCRIPT = 'import gc, json, os, sys, time, torch\nfrom omnilingual_asr.models.inference.pipeline import ASRInferencePipeline\n\ncards, recordings = sys.argv[1].split(","), json.load(open("recordings.json"))\ngpu = torch.cuda.get_device_name(0)\nresults = {}\nfor card in cards:\n    t0 = time.time()\n    pipeline = ASRInferencePipeline(model_card=card)  # bf16 on the GPU by default\n    print(f"{card}: loaded in {time.time() - t0:.0f} s", flush=True)\n    transcripts = []\n    for r in recordings:\n        t1 = time.time()\n        text = pipeline.transcribe([r["path"]], lang=["swh_Latn"], batch_size=1)[0]\n        transcripts.append({"survey_id": r["survey_id"], "question_id": r["question_id"],\n                            "wav_file": os.path.basename(r["path"]), "transcript": text,\n                            "seconds": round(time.time() - t1, 1)})\n        print(f"  {r[\'survey_id\'][:8]} {r[\'question_id\']:>3}: {text}", flush=True)\n    results[card] = {"model_file": card, "transcripts": transcripts}\n    del pipeline; gc.collect(); torch.cuda.empty_cache()\n\nout = {"description": "omniASR transcripts of the 30 office_testing recordings (expected_answers.json).",\n       "runtime": f"omnilingual-asr on Google Colab, {gpu}, bf16, torch {torch.__version__}",\n       "decoding": "ASRInferencePipeline defaults, lang swh_Latn, batch_size 1",\n       "models": results}\njson.dump(out, open("omniasr_transcripts.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)\nprint("Saved omniasr_transcripts.json")\n'
open('transcribe.py', 'w').write(SCRIPT)
json.dump(RECORDINGS, open('recordings.json', 'w'))
cards = ','.join(MODEL_CARDS)
!/content/omni/bin/python transcribe.py {cards}

In [ ]:
# 5. Download the results
from google.colab import files
files.download("omniasr_transcripts.json")